In [22]:
import pandas as pd
from CCA_utils import *

## Baseline Model Panel

In [28]:
study_sovereigns = [
    'Saudi Arabia', 'Abu Dhabi', 'Dubai', 'Qatar', 'Colombia',
    'Mexico', 'Brazil', 'Egypt', 'Malaysia','Indonesia', 'Philippines', 'Turkey', 'Chile', 'China',
    'South Africa', 'South Korea', 'Thailand']

cca_panel_df = pd.read_csv('../data/processed/CCA/cca_newfx_rates.csv')
cca_panel_df['date'] = pd.to_datetime(cca_panel_df['date'])
cca_panel_df = cca_panel_df[cca_panel_df['country'].isin(study_sovereigns)].copy()
cca_panel_df.drop(columns=['cds_spread_1Y'], inplace=True)
cca_panel_df.rename(columns={'cds_spread_5Y': 'cds_spread'}, inplace=True)


T=5.0
vol_window = 52
freq = 'W'

cca_panel_df['domestic_rate_in_units'] = cca_panel_df['domestic_rate_in_units']/100
cca_panel_df['risk_free_rate'] = cca_panel_df['risk_free_rate']/100
cca_panel_df['monetary_base_mn_localcurr'] = cca_panel_df['monetary_base_mn_localcurr'] / 1000
cca_panel_df['domestic_debt_bn_localcurr'] = cca_panel_df['domestic_debt_bn_localcurr']
cca_panel_df['external_debt_mn_usd'] = cca_panel_df['external_debt_mn_usd']/1000

## Model 1: Specific data

In [29]:

#Join oil prices and oil futures
oil_prices = pd.read_csv('../data/processed/Oil/oil_prices_datastream.csv')
oil_prices['date'] = pd.to_datetime(oil_prices['date'])
oil_futures = pd.read_csv('../data/processed/Oil/oil_futures.csv')
oil_futures['date'] = pd.to_datetime(oil_futures['date'])

cca_panel_df = cca_panel_df.merge(oil_prices[['date','Brent']], on='date', how='left')
cca_panel_df = cca_panel_df.merge(oil_futures[['date','Brent_12m']],on='date',how='left')

oil_prod = pd.read_csv('../data/processed/Oil/crude_oil_production.csv')
oil_prod = oil_prod.rename(columns={oil_prod.columns[0]: 'country'})
oil_prod = oil_prod.melt(id_vars='country', var_name='year', value_name='oil_production_Mt_yr')
oil_prod['year'] = oil_prod['year'].astype(int)

# UAE for Abu Dhabi and Dubai
uae = oil_prod[oil_prod['country'] == 'United Arab Emirates'].copy()
for name in ['Abu Dhabi', 'Dubai']:
    rows = uae.copy()
    rows['country'] = name
    oil_prod = pd.concat([oil_prod, rows], ignore_index=True)

cca_panel_df['year'] = cca_panel_df['date'].dt.year
cca_panel_df = cca_panel_df.merge(oil_prod[['year', 'country', 'oil_production_Mt_yr']], 
                                   on=['country', 'year'], how='left')
cca_panel_df['oil_production_Mt_yr'] = cca_panel_df['oil_production_Mt_yr'].fillna(0)
cca_panel_df['oil_production_Mt_yr'] = cca_panel_df['oil_production_Mt_yr'].astype(int)

cca_panel_df.set_index(['date','country'], inplace=True)
cca_panel_df = (
    cca_panel_df
    .groupby('country')
    .resample(freq, level='date')
    .last()
)
cca_panel_df.reset_index(inplace=True)
cca_panel_df

C:\Users\JuanFranciscoPerez\AppData\Local\Temp\ipykernel_12280\1962628297.py:3: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  oil_prices['date'] = pd.to_datetime(oil_prices['date'])


C:\Users\JuanFranciscoPerez\AppData\Local\Temp\ipykernel_12280\1962628297.py:5: UserWarning: Parsing dates in %d.%m.%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  oil_futures['date'] = pd.to_datetime(oil_futures['date'])


,country,date,Unnamed: 0,year,quarter,month,week,cds_spread,msci_index,monetary_base_mn_localcurr,domestic_debt_bn_localcurr,external_debt_mn_usd,domestic_rate_in_units,risk_free_rate,fx_rate,Brent,Brent_12m,oil_production_Mt_yr
0,Abu Dhabi,2014-01-05,13776,2014,1.0,1.0,1.0,56.31999,749.581,288.310000,210.154,22.889480,0.01,0.0013,3.672960,107.07,102.72,170
1,Abu Dhabi,2014-01-12,13777,2014,1.0,1.0,2.0,55.85999,771.135,288.310000,210.154,22.889480,0.01,0.0012,3.672960,106.33,102.87,170
2,Abu Dhabi,2014-01-19,13778,2014,1.0,1.0,3.0,55.32999,777.754,288.310000,210.154,22.889480,0.01,0.0011,3.672960,106.89,102.13,170
3,Abu Dhabi,2014-01-26,13779,2014,1.0,1.0,4.0,55.34999,808.655,288.310000,210.154,22.889480,0.01,0.0011,3.672960,107.45,102.91,170
4,Abu Dhabi,2014-02-02,13780,2014,1.0,1.0,5.0,56.32999,804.960,288.310000,210.154,22.889480,0.01,0.0010,3.672960,107.13,101.63,170
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9753,Turkey,2024-12-01,13197,2024,4.0,11.0,48.0,251.12000,610.455,6067.888530,10716.668,157.127935,0.50,0.0430,34.722222,73.20,69.95,6
9754,Turkey,2024-12-08,13198,2024,4.0,12.0,49.0,241.90000,610.777,6585.275534,10716.668,157.127935,0.50,0.0419,34.722222,72.19,69.35,6
9755,Turkey,2024-12-15,13199,2024,4.0,12.0,50.0,247.11000,644.678,6585.275534,10716.668,157.127935,0.50,0.0424,34.965035,74.16,71.67,6
9756,Turkey,2024-12-22,13200,2024,4.0,12.0,51.0,256.52000,626.599,6585.275534,10716.668,157.127935,0.50,0.0427,35.211268,72.83,69.94,6


In [30]:
results = pd.DataFrame()
gamma = 0.30

for country, group in cca_panel_df.groupby('country'):

    df = group.copy().sort_values('date').reset_index(drop=True)
        
    # --- Unit conversions ---
    r_d = df['domestic_rate_in_units']
    r_f = df['risk_free_rate']
    M_bn = df['monetary_base_mn_localcurr']
    dom_D_bn = df['domestic_debt_bn_localcurr']
    ext_D_bn = df['external_debt_mn_usd']
    fx_rate = df['fx_rate']


    # --- LCL$ ---
    df['LCL_usd'] = [
        compute_lcl_usd(m, bd, fx, rd, rf, T)
        for m, bd, fx, rd, rf in zip(
            M_bn, dom_D_bn, fx_rate, r_d, r_f
        )
    ]

        # --- Barrier ---
    df['B_f'] = [
            compute_barrier_kvm(debt, rf, T)
            for debt, rf in zip(    
                df['external_debt_mn_usd'], r_f
            )
        ]

    df['futures_slope'] = df['Brent'] / df['Brent_12m']
    futures_slope= 1 + gamma *(df['futures_slope'] - 1)

    df['LCL_aug'] = df['LCL_usd'] * futures_slope
    ann_factor = np.sqrt(52) if freq == 'W' else np.sqrt(12)
    log_ret = np.log(df['LCL_aug'] / df['LCL_aug'].shift(1))



    df['sigma_lcl'] = log_ret.rolling(window=vol_window).std() * ann_factor  # use aug, not base

    # --- Solve CCA with augmented LCL ---
    out = {k: [] for k in ['implied_V', 'implied_sigma_V', 'cca_converged',
                            'distance_to_distress', 'default_prob',
                            'model_spread_bps', 'put_value', 'risky_debt',
                            'leverage']}

    for i, row in df.iterrows():
        cca = solve_CCA(row['LCL_aug'], row['sigma_lcl'], row['B_f'],
                        r_f.iloc[i], T)
        risk = compute_risk(cca['V'], cca['sigma_V'], row['B_f'],
                           r_f.iloc[i], T)

        out['implied_V'].append(cca['V'])
        out['implied_sigma_V'].append(cca['sigma_V'])
        out['cca_converged'].append(cca['converged'])
        out['distance_to_distress'].append(risk['d2'])
        out['default_prob'].append(risk['default_prob'])
        out['model_spread_bps'].append(risk['credit_spread_bps'])
        out['put_value'].append(risk['put_value'])
        out['risky_debt'].append(risk['risky_debt'])
        out['leverage'].append(risk['leverage'])

    for col, vals in out.items():
        df[col] = vals

    results = pd.concat([results, df])


START_DATE = '2015-01-01'
END_DATE = '2024-12-31'

results = results[
    (results['date'] >= START_DATE) & (results['date'] <= END_DATE)
].copy()


In [31]:
results.to_csv("../output/results/M1_results_5YCDS.csv")